# Results paragraph: Quality effects (IQM winner-take-all and susceptibility)

Uses **age–quality effects** (harmonized, non-vendorwise): for each bundle × metric × IQM we have qc_effect_size (ΔR²adj). Winner-take-all: per (bundle, metric) which IQM had the largest qc_effect_size; count wins by IQM category. Illustrative IQM = preprocessed dMRI contrast (t1post_dwi_contrast, after B1 bias correction) for FA vs ICVF susceptibility. Data: `data/age_quality_effects/age_quality_effects_all_outputs.rds`. **Fig. 6:** Panel A = winner-take-all bar chart; Panel B/C = t1post_dwi_contrast scatter and heatmap.

## Setup and load quality-effect data

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(fs)
  library(jsonlite)
  library(stringr)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

age_rds <- fs::path(project_root, "data", "age_quality_effects", "age_quality_effects_all_outputs.rds")
if (!file.exists(age_rds)) stop("Age effects file not found: ", age_rds)

df_all <- readRDS(age_rds)
if (!is.data.frame(df_all)) stop("Age effects file is not a data.frame.")

metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT", NODDI_icvf = "ICVF", MAPMRI_rtop = "RTOP",
  GQI_fa = "FA", GQI_md = "MD"
)

iqm_metrics <- c(
  "raw_neighbor_corr", "raw_masked_neighbor_corr", "raw_dwi_contrast", "raw_num_bad_slices",
  "raw_coherence_index", "raw_incoherence_index",
  "t1_neighbor_corr", "t1_masked_neighbor_corr", "t1_dwi_contrast", "t1_num_bad_slices",
  "t1_coherence_index", "t1_incoherence_index",
  "t1post_neighbor_corr", "t1post_masked_neighbor_corr", "t1post_dwi_contrast", "t1post_num_bad_slices",
  "t1post_coherence_index", "t1post_incoherence_index",
  "mean_fd", "max_fd", "max_rotation", "max_translation", "max_rel_rotation", "max_rel_translation", "t1_dice_distance",
  "CNR0_mean", "CNR1_mean", "CNR2_mean", "CNR3_mean", "CNR4_mean",
  "CNR0_median", "CNR1_median", "CNR2_median", "CNR3_median", "CNR4_median",
  "CNR0_standard_deviation", "CNR1_standard_deviation", "CNR2_standard_deviation", "CNR3_standard_deviation", "CNR4_standard_deviation"
)

df_qc <- df_all %>%
  filter(
    .data[["output_type"]] == "non_vendorwise_pairwise",
    .data[["source"]] == "harmonized",
    metric %in% metrics_keep,
    .data[["qc_metric"]] %in% iqm_metrics,
    .data[["qc_metric"]] != "qc_prediction",
    !is.na(bundle),
    !is.na(.data[["qc_effect_size"]])
  ) %>%
  transmute(
    bundle = bundle,
    metric = metric,
    metric_label = unname(metric_labels[metric]),
    iqm = .data[["qc_metric"]],
    qc_effect_size = as.numeric(.data[["qc_effect_size"]])
  )

if (nrow(df_qc) == 0) stop("No quality-effect rows found.")
cat("N quality-effect rows (harmonized, five metrics, IQM covariates):", nrow(df_qc), "\n")

## Winner-take-all counts and t1post_dwi_contrast susceptibility

1. For each (bundle, metric), which IQM had the largest qc_effect_size? Count wins by IQM.  
2. Paragraph categories: preprocessed dMRI contrast (t1post_dwi_contrast); neighboring dMRI correlation (all neighbor_corr variants); preprocessed incoherence pre-B1 (t1_incoherence_index).  
3. For t1post_dwi_contrast: effect size range and mean by metric; FA = most susceptible, ICVF = most robust.

In [ ]:
wta <- df_qc %>%
  group_by(metric, bundle) %>%
  slice_max(order_by = qc_effect_size, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  count(iqm, name = "n_wins")

n_dwi_contrast <- wta %>% filter(iqm == "t1post_dwi_contrast") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)
neighbor_iqms <- c("raw_neighbor_corr", "raw_masked_neighbor_corr", "t1_neighbor_corr", "t1_masked_neighbor_corr", "t1post_neighbor_corr", "t1post_masked_neighbor_corr")
n_neighbor <- wta %>% filter(iqm %in% neighbor_iqms) %>% pull(n_wins) %>% sum(., na.rm = TRUE)
n_incoherence <- wta %>% filter(iqm == "t1_incoherence_index") %>% pull(n_wins) %>% replace(., length(.) == 0, 0L) %>% `[`(1)

dwi_contrast_effects <- df_qc %>%
  filter(iqm == "t1post_dwi_contrast") %>%
  group_by(metric_label) %>%
  summarise(
    min_effect = min(qc_effect_size, na.rm = TRUE),
    max_effect = max(qc_effect_size, na.rm = TRUE),
    mean_effect = mean(qc_effect_size, na.rm = TRUE),
    .groups = "drop"
  )

fa_row   <- dwi_contrast_effects %>% filter(metric_label == "FA")   %>% slice(1)
icvf_row <- dwi_contrast_effects %>% filter(metric_label == "ICVF") %>% slice(1)

effect_as_pct <- function(x) round(x * 100, 1)
fa_range_lo   <- effect_as_pct(fa_row$min_effect)
fa_range_hi   <- effect_as_pct(fa_row$max_effect)
fa_mean_pct   <- effect_as_pct(fa_row$mean_effect)
icvf_range_lo <- effect_as_pct(icvf_row$min_effect)
icvf_range_hi <- effect_as_pct(icvf_row$max_effect)
icvf_mean_pct <- effect_as_pct(icvf_row$mean_effect)

fa_range_str   <- paste0(fa_range_lo, "-", fa_range_hi)
icvf_range_str <- paste0(icvf_range_lo, "-", icvf_range_hi)

cat("Winner-take-all counts (bundle–metric combinations):\n")
cat("  Preprocessed dMRI contrast (t1post_dwi_contrast):", n_dwi_contrast, "\n")
cat("  Neighboring dMRI correlation (all variants):", n_neighbor, "\n")
cat("  Preprocessed incoherence index (t1_incoherence_index, pre-B1):", n_incoherence, "\n")
cat("\nt1post_dwi_contrast effect size (%% variance) by metric:\n")
print(dwi_contrast_effects %>% mutate(across(c(min_effect, max_effect, mean_effect), ~ round(. * 100, 2))))
cat("\nFA range:", fa_range_str, "%; mean:", fa_mean_pct, "%\n")
cat("ICVF range:", icvf_range_str, "%; mean:", icvf_mean_pct, "%\n")

## Filled paragraph

In [ ]:
part1 <- paste("Across the five microstructural metrics and 62 bundles, we identified the IQM yielding the largest effect size for each bundle–metric combination. The preprocessed dMRI contrast (after B1 bias correction) accounted for the greatest variance in", n_dwi_contrast, "bundle–metric combinations. Measures of neighboring dMRI correlation (including masked and unmasked, raw and preprocessed variants) accounted for the greatest variance in", n_neighbor, "combinations.")
part2 <- paste("The incoherence index of the preprocessed data (pre-B1 correction) accounted for the greatest variance in", n_incoherence, "combinations (Fig. 6a).")
part3 <- paste("Overall, susceptibility to image quality was modest but heterogeneous across both microstructural metrics and bundles (Fig. 6b). Using preprocessed dMRI contrast as an illustrative IQM, FA was, on average, the most susceptible to variation in image quality (range of effect sizes across bundles:", fa_range_str, "%; average effect size:", fa_mean_pct, "%).")
part4 <- paste("In contrast, ICVF was the most robust to image quality (range of effect sizes across bundles:", icvf_range_str, "%; average effect size:", icvf_mean_pct, "%).")
part5 <- "Together, these findings indicate that IQMs capturing nuanced diffusion q-space properties most strongly explain variability in microstructural estimates. Commonly used proxies such as head motion alone may provide an incomplete characterization of data quality. Moreover, fractional anisotropy (FA) appears more sensitive to image quality variation than advanced multi-shell diffusion metrics."
txt <- paste(part1, part2, part3, part4, part5, sep = " ")
cat(txt, "\n")